# Extract explainable timbre targets from MTG-Jamendo audio

This notebook reads the MP3 files previously stored in Google Drive and creates one consolidated CSV row for every track in `split_csv.csv`.

The output contains the agreed **35 raw timbre descriptors**:

- spectral centroid mean and standard deviation (2);
- spectral bandwidth mean and standard deviation (2);
- spectral contrast mean (1);
- spectral flatness mean (1);
- 85% spectral roll-off mean (1);
- harmonic-to-noise ratio mean (1);
- normalized inharmonicity mean (1);
- MFCC 1-13 means and standard deviations (26).

The notebook is resumable. Every successfully processed track is appended to a checkpoint CSV in Drive. Failed tracks are recorded separately and may be retried.

> The output remains unstandardized. Fit the feature scaler on the training partition only when training the timbre head; fitting it here on all tracks would leak validation/test distribution information.


In [ ]:
# Install a reproducible audio-analysis environment.
!pip -q install "librosa==0.10.2.post1" "soundfile==0.12.1" "audioread==3.0.1" "tqdm>=4.66"


In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Configuration.
from pathlib import Path, PurePosixPath
import csv
import hashlib
import json
import math
import os
import re
import shutil
import time
import warnings

import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/content/drive/MyDrive/MTG_Jamendo_7324_full_quality')
CSV_PATH = PROJECT_ROOT / 'split_csv.csv'
AUDIO_ROOT = PROJECT_ROOT / 'audio'
FEATURE_ROOT = PROJECT_ROOT / 'timbre_features'

CHECKPOINT_CSV = FEATURE_ROOT / 'timbre_features_checkpoint.csv'
FINAL_CSV = FEATURE_ROOT / 'timbre_features_raw.csv'
ERROR_CSV = FEATURE_ROOT / 'timbre_feature_errors.csv'
CONFIG_JSON = FEATURE_ROOT / 'timbre_extraction_config.json'

# Audio analysis configuration.
SAMPLE_RATE = 22_050
N_FFT = 2_048
HOP_LENGTH = 512
ROLLOFF_PERCENT = 0.85
N_MFCC = 13

# Long tracks are represented by uniformly distributed windows.
# Set NUM_WINDOWS=None to analyze every complete track (much slower).
NUM_WINDOWS = 6
WINDOW_SECONDS = 15.0

# HNR and inharmonicity use larger waveform frames and a bounded sample of frames.
HARMONIC_FRAME_LENGTH = 4_096
HARMONIC_HOP_LENGTH = 1_024
FMIN = 50.0
FMAX = 2_000.0
MAX_HARMONIC_FRAMES_PER_WINDOW = 80
MIN_RMS_DB_BELOW_PEAK = 45.0
MIN_PERIODICITY = 0.05

# Save progress frequently to Drive. Sequential processing is deliberately used
# because mounted Drive I/O is usually the bottleneck and it is easier to resume safely.
RETRY_FAILED_TRACKS = True
LIMIT_TRACKS = None  # e.g. 10 for a small trial; None processes every remaining track.

FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = (
    ['spectral_centroid_mean', 'spectral_centroid_std',
     'spectral_bandwidth_mean', 'spectral_bandwidth_std',
     'spectral_contrast_mean', 'spectral_flatness_mean',
     'spectral_rolloff_mean', 'hnr_mean_db', 'inharmonicity_mean']
    + [f'mfcc_{i:02d}_mean' for i in range(1, 14)]
    + [f'mfcc_{i:02d}_std' for i in range(1, 14)]
)

assert len(FEATURE_COLUMNS) == 35
print('Project root:', PROJECT_ROOT)
print('Feature count:', len(FEATURE_COLUMNS))


In [ ]:
# Load and validate metadata and audio paths.
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Missing metadata file: {CSV_PATH}')
if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f'Missing audio directory: {AUDIO_ROOT}')

metadata = pd.read_csv(CSV_PATH, dtype={'TRACK_ID': str, 'PATH': str})
required_columns = {'TRACK_ID', 'PATH', 'DURATION'}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

def normalize_relative_path(value):
    value = str(value).replace('\\', '/').lstrip('/')
    parts = PurePosixPath(value).parts
    if len(parts) != 2 or not re.fullmatch(r'\d{2}', parts[0]) or not parts[1].lower().endswith('.mp3'):
        raise ValueError(f'Unexpected MTG-Jamendo path: {value!r}')
    return f'{parts[0]}/{parts[1]}'

metadata['REL_PATH'] = metadata['PATH'].map(normalize_relative_path)
if metadata['TRACK_ID'].duplicated().any():
    raise ValueError('TRACK_ID must be unique')
if metadata['REL_PATH'].duplicated().any():
    raise ValueError('PATH must be unique')

metadata['AUDIO_PATH'] = metadata['REL_PATH'].map(lambda p: str(AUDIO_ROOT / p))
metadata['AUDIO_EXISTS'] = metadata['AUDIO_PATH'].map(lambda p: Path(p).exists())

print(f'Metadata tracks: {len(metadata):,}')
print(f'Audio files found: {metadata.AUDIO_EXISTS.sum():,}')
print(f'Audio files missing: {(~metadata.AUDIO_EXISTS).sum():,}')
if not metadata['AUDIO_EXISTS'].all():
    display(metadata.loc[~metadata['AUDIO_EXISTS'], ['TRACK_ID', 'REL_PATH']].head(20))
    raise FileNotFoundError('Some requested MP3 files are missing. Complete the audio download first.')


In [ ]:
# Feature-extraction functions.
def finite_values(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return values[np.isfinite(values)]

def safe_mean(values):
    values = finite_values(values)
    return float(np.mean(values)) if values.size else np.nan

def safe_std(values):
    values = finite_values(values)
    return float(np.std(values, ddof=0)) if values.size else np.nan

def uniform_window_offsets(duration, window_seconds, num_windows):
    duration = float(duration)
    if num_windows is None or duration <= window_seconds:
        return [0.0], None if num_windows is None else min(duration, window_seconds)
    last_start = max(0.0, duration - window_seconds)
    offsets = np.linspace(0.0, last_start, int(num_windows), dtype=np.float64)
    # Rounding prevents effectively duplicated offsets on short tracks.
    return sorted(set(float(round(x, 3)) for x in offsets)), window_seconds

def select_valid_frame_indices(frames, maximum):
    rms = np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1) + 1e-12)
    peak = float(np.max(rms)) if rms.size else 0.0
    if peak <= 0:
        return np.array([], dtype=int)
    threshold = peak * (10.0 ** (-MIN_RMS_DB_BELOW_PEAK / 20.0))
    valid = np.flatnonzero(rms >= threshold)
    if valid.size > maximum:
        positions = np.linspace(0, valid.size - 1, maximum).round().astype(int)
        valid = valid[positions]
    return valid

def harmonic_descriptors(y, sr):
    """Estimate HNR and normalized inharmonicity from valid quasi-periodic frames.

    HNR is based on normalized autocorrelation at the YIN-estimated pitch period.
    Inharmonicity is the magnitude-weighted relative deviation of local spectral
    peaks from integer multiples of the estimated fundamental. Both are track-level
    timbre descriptors; the estimated pitch itself is never retained.
    """
    y = np.asarray(y, dtype=np.float32)
    if y.size < HARMONIC_FRAME_LENGTH:
        return np.array([]), np.array([])

    frames = librosa.util.frame(
        y, frame_length=HARMONIC_FRAME_LENGTH,
        hop_length=HARMONIC_HOP_LENGTH
    ).T.copy()
    valid_indices = select_valid_frame_indices(frames, MAX_HARMONIC_FRAMES_PER_WINDOW)
    if not valid_indices.size:
        return np.array([]), np.array([])

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f0_all = librosa.yin(
            y, fmin=FMIN, fmax=min(FMAX, sr / 2 - 1), sr=sr,
            frame_length=HARMONIC_FRAME_LENGTH,
            hop_length=HARMONIC_HOP_LENGTH,
            center=False
        )

    window = np.hanning(HARMONIC_FRAME_LENGTH).astype(np.float32)
    frequencies = np.fft.rfftfreq(HARMONIC_FRAME_LENGTH, d=1.0 / sr)
    hnr_values = []
    inharmonicity_values = []

    for frame_index in valid_indices:
        if frame_index >= len(f0_all):
            continue
        f0 = float(f0_all[frame_index])
        if not np.isfinite(f0) or f0 < FMIN or f0 > FMAX:
            continue

        frame = frames[frame_index].astype(np.float64)
        frame -= np.mean(frame)
        frame *= window
        lag = int(round(sr / f0))
        if lag < 1 or lag >= frame.size - 1:
            continue

        left = frame[:-lag]
        right = frame[lag:]
        denominator = math.sqrt(float(np.dot(left, left) * np.dot(right, right))) + 1e-12
        periodicity = float(np.dot(left, right) / denominator)
        if not np.isfinite(periodicity) or periodicity <= MIN_PERIODICITY:
            continue
        periodicity = float(np.clip(periodicity, 1e-6, 1.0 - 1e-6))
        hnr_values.append(10.0 * math.log10(periodicity / (1.0 - periodicity)))

        spectrum = np.abs(np.fft.rfft(frame))
        deviations = []
        weights = []
        harmonic_number = 1
        while harmonic_number * f0 < sr / 2:
            expected_frequency = harmonic_number * f0
            expected_bin = int(round(expected_frequency * HARMONIC_FRAME_LENGTH / sr))
            lower = max(1, expected_bin - 2)
            upper = min(len(spectrum) - 1, expected_bin + 2)
            if lower > upper:
                break
            local_bin = lower + int(np.argmax(spectrum[lower:upper + 1]))
            magnitude = float(spectrum[local_bin])
            if magnitude > 0:
                observed_frequency = float(frequencies[local_bin])
                deviations.append(abs(observed_frequency - expected_frequency) / expected_frequency)
                weights.append(magnitude)
            harmonic_number += 1
        if weights and np.sum(weights) > 0:
            inharmonicity_values.append(float(np.average(deviations, weights=weights)))

    return np.asarray(hnr_values), np.asarray(inharmonicity_values)

def extract_timbre_features(audio_path, reported_duration=None):
    audio_path = Path(audio_path)
    try:
        measured_duration = float(librosa.get_duration(path=str(audio_path)))
    except Exception:
        measured_duration = float(reported_duration)
    if not np.isfinite(measured_duration) or measured_duration <= 0:
        raise ValueError(f'Invalid duration for {audio_path}')

    offsets, load_duration = uniform_window_offsets(measured_duration, WINDOW_SECONDS, NUM_WINDOWS)
    centroid_values, bandwidth_values = [], []
    contrast_values, flatness_values, rolloff_values = [], [], []
    mfcc_values, hnr_values, inharmonicity_values = [], [], []
    analyzed_seconds = 0.0

    for offset in offsets:
        y, sr = librosa.load(
            str(audio_path), sr=SAMPLE_RATE, mono=True,
            offset=float(offset), duration=load_duration,
            res_type='soxr_hq'
        )
        y = np.asarray(y, dtype=np.float32)
        if y.size < N_FFT:
            continue
        if not np.any(np.isfinite(y)):
            continue
        y = np.nan_to_num(y, copy=False)
        analyzed_seconds += y.size / sr

        magnitude = np.abs(librosa.stft(
            y, n_fft=N_FFT, hop_length=HOP_LENGTH,
            window='hann', center=True
        ))
        power = magnitude ** 2
        centroid_values.append(librosa.feature.spectral_centroid(S=magnitude, sr=sr)[0])
        bandwidth_values.append(librosa.feature.spectral_bandwidth(S=magnitude, sr=sr)[0])
        contrast_values.append(librosa.feature.spectral_contrast(S=magnitude, sr=sr).reshape(-1))
        flatness_values.append(librosa.feature.spectral_flatness(S=power)[0])
        rolloff_values.append(librosa.feature.spectral_rolloff(
            S=magnitude, sr=sr, roll_percent=ROLLOFF_PERCENT
        )[0])
        mfcc_values.append(librosa.feature.mfcc(
            y=y, sr=sr, n_mfcc=N_MFCC,
            n_fft=N_FFT, hop_length=HOP_LENGTH
        ))
        hnr, inharmonicity = harmonic_descriptors(y, sr)
        hnr_values.append(hnr)
        inharmonicity_values.append(inharmonicity)

    if not centroid_values or not mfcc_values:
        raise ValueError(f'No valid analysis windows for {audio_path}')

    centroid = np.concatenate(centroid_values)
    bandwidth = np.concatenate(bandwidth_values)
    contrast = np.concatenate(contrast_values)
    flatness = np.concatenate(flatness_values)
    rolloff = np.concatenate(rolloff_values)
    mfcc = np.concatenate(mfcc_values, axis=1)
    hnr = np.concatenate(hnr_values) if hnr_values else np.array([])
    inharmonicity = np.concatenate(inharmonicity_values) if inharmonicity_values else np.array([])

    features = {
        'spectral_centroid_mean': safe_mean(centroid),
        'spectral_centroid_std': safe_std(centroid),
        'spectral_bandwidth_mean': safe_mean(bandwidth),
        'spectral_bandwidth_std': safe_std(bandwidth),
        'spectral_contrast_mean': safe_mean(contrast),
        'spectral_flatness_mean': safe_mean(flatness),
        'spectral_rolloff_mean': safe_mean(rolloff),
        'hnr_mean_db': safe_mean(hnr),
        'inharmonicity_mean': safe_mean(inharmonicity),
    }
    for index in range(N_MFCC):
        features[f'mfcc_{index + 1:02d}_mean'] = safe_mean(mfcc[index])
    for index in range(N_MFCC):
        features[f'mfcc_{index + 1:02d}_std'] = safe_std(mfcc[index])

    if list(features) != FEATURE_COLUMNS:
        raise AssertionError('Feature order does not match the declared 35-D contract')
    return features, {
        'measured_duration_sec': measured_duration,
        'analyzed_duration_sec': analyzed_seconds,
        'analysis_windows': len(offsets),
        'hnr_valid_frames': int(finite_values(hnr).size),
        'inharmonicity_valid_frames': int(finite_values(inharmonicity).size),
    }

print('Extraction functions ready.')


In [ ]:
# Synthetic smoke test. Run this before processing the Drive collection.
# It validates dimensionality, finiteness of the core spectral/MFCC descriptors,
# feature ordering, and end-to-end decoding/extraction.
import soundfile as sf

SMOKE_DIR = Path('/content/timbre_smoke_test')
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_WAV = SMOKE_DIR / 'synthetic_timbre.wav'

duration = 8.0
t = np.arange(int(SAMPLE_RATE * duration), dtype=np.float32) / SAMPLE_RATE
envelope = np.minimum(1.0, t / 0.08) * np.exp(-0.04 * t)
rng = np.random.default_rng(42)
signal = envelope * (
    0.65 * np.sin(2 * np.pi * 220.0 * t)
    + 0.25 * np.sin(2 * np.pi * 440.7 * t)
    + 0.12 * np.sin(2 * np.pi * 663.0 * t)
    + 0.01 * rng.standard_normal(t.size)
)
signal = (signal / (np.max(np.abs(signal)) + 1e-9)).astype(np.float32)
sf.write(SMOKE_WAV, signal, SAMPLE_RATE)

smoke_features, smoke_quality = extract_timbre_features(SMOKE_WAV, duration)
assert len(smoke_features) == 35
assert list(smoke_features) == FEATURE_COLUMNS
core_names = [name for name in FEATURE_COLUMNS if name not in {'hnr_mean_db', 'inharmonicity_mean'}]
assert all(np.isfinite(smoke_features[name]) for name in core_names)
assert smoke_quality['analyzed_duration_sec'] > 0
print('Smoke test passed: 35 ordered features extracted.')
display(pd.DataFrame([smoke_features]).T.rename(columns={0: 'value'}))
print(smoke_quality)


In [ ]:
# Save the exact extraction configuration with the output.
configuration = {
    'sample_rate': SAMPLE_RATE,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'rolloff_percent': ROLLOFF_PERCENT,
    'n_mfcc': N_MFCC,
    'num_windows': NUM_WINDOWS,
    'window_seconds': WINDOW_SECONDS,
    'harmonic_frame_length': HARMONIC_FRAME_LENGTH,
    'harmonic_hop_length': HARMONIC_HOP_LENGTH,
    'fmin': FMIN,
    'fmax': FMAX,
    'max_harmonic_frames_per_window': MAX_HARMONIC_FRAMES_PER_WINDOW,
    'min_rms_db_below_peak': MIN_RMS_DB_BELOW_PEAK,
    'min_periodicity': MIN_PERIODICITY,
    'feature_columns': FEATURE_COLUMNS,
    'hnr_definition': 'normalized-autocorrelation HNR at the YIN-estimated period',
    'inharmonicity_definition': 'magnitude-weighted relative partial deviation from integer harmonics',
    'librosa_version': librosa.__version__,
    'numpy_version': np.__version__,
}
CONFIG_JSON.write_text(json.dumps(configuration, indent=2), encoding='utf-8')
print('Saved:', CONFIG_JSON)


In [ ]:
# Resumable extraction loop.
IDENTITY_COLUMNS = ['TRACK_ID', 'REL_PATH', 'reported_duration_sec']
QUALITY_COLUMNS = [
    'measured_duration_sec', 'analyzed_duration_sec', 'analysis_windows',
    'hnr_valid_frames', 'inharmonicity_valid_frames', 'extraction_status'
]
OUTPUT_COLUMNS = IDENTITY_COLUMNS + FEATURE_COLUMNS + QUALITY_COLUMNS

if CHECKPOINT_CSV.exists():
    checkpoint = pd.read_csv(CHECKPOINT_CSV, dtype={'TRACK_ID': str, 'REL_PATH': str})
    completed_ids = set(checkpoint.loc[checkpoint['extraction_status'] == 'ok', 'TRACK_ID'])
    print(f'Resuming with {len(completed_ids):,} successful tracks already stored.')
else:
    completed_ids = set()

if ERROR_CSV.exists() and RETRY_FAILED_TRACKS:
    ERROR_CSV.unlink()

pending = metadata.loc[~metadata['TRACK_ID'].isin(completed_ids)].copy()
if LIMIT_TRACKS is not None:
    pending = pending.head(int(LIMIT_TRACKS))
print(f'Tracks pending in this run: {len(pending):,}')

checkpoint_needs_header = not CHECKPOINT_CSV.exists() or CHECKPOINT_CSV.stat().st_size == 0
error_needs_header = not ERROR_CSV.exists() or ERROR_CSV.stat().st_size == 0

with CHECKPOINT_CSV.open('a', newline='', encoding='utf-8') as checkpoint_file, \
     ERROR_CSV.open('a', newline='', encoding='utf-8') as error_file:
    checkpoint_writer = csv.DictWriter(checkpoint_file, fieldnames=OUTPUT_COLUMNS)
    error_writer = csv.DictWriter(error_file, fieldnames=['TRACK_ID', 'REL_PATH', 'error_type', 'error_message'])
    if checkpoint_needs_header:
        checkpoint_writer.writeheader()
    if error_needs_header:
        error_writer.writeheader()

    for row in tqdm(pending.itertuples(index=False), total=len(pending), desc='Extract timbre'):
        try:
            features, quality = extract_timbre_features(row.AUDIO_PATH, row.DURATION)
            output = {
                'TRACK_ID': row.TRACK_ID,
                'REL_PATH': row.REL_PATH,
                'reported_duration_sec': float(row.DURATION),
                **features,
                **quality,
                'extraction_status': 'ok',
            }
            checkpoint_writer.writerow(output)
            checkpoint_file.flush()
        except Exception as exc:
            error_writer.writerow({
                'TRACK_ID': row.TRACK_ID,
                'REL_PATH': row.REL_PATH,
                'error_type': type(exc).__name__,
                'error_message': str(exc)[:1000],
            })
            error_file.flush()

print('Extraction run finished. The checkpoint remains safe in Drive.')


In [ ]:
# Finalize and verify the consolidated CSV.
if not CHECKPOINT_CSV.exists():
    raise FileNotFoundError('No extraction checkpoint was produced')

features_df = pd.read_csv(CHECKPOINT_CSV, dtype={'TRACK_ID': str, 'REL_PATH': str})
# If a track was retried, keep its latest successful result.
features_df = features_df.loc[features_df['extraction_status'] == 'ok'].copy()
features_df = features_df.drop_duplicates('TRACK_ID', keep='last')

expected_ids = set(metadata['TRACK_ID'])
completed_ids = set(features_df['TRACK_ID'])
missing_ids = expected_ids - completed_ids
unexpected_ids = completed_ids - expected_ids

if unexpected_ids:
    raise AssertionError(f'Output contains {len(unexpected_ids)} unexpected TRACK_ID values')
if list(features_df[FEATURE_COLUMNS].columns) != FEATURE_COLUMNS:
    raise AssertionError('The final feature order differs from the 35-D contract')

# Ensure all non-missing numeric values are finite.
numeric = features_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
if np.isinf(numeric).any():
    raise AssertionError('Infinite feature values detected')

order = {track_id: index for index, track_id in enumerate(metadata['TRACK_ID'])}
features_df['_order'] = features_df['TRACK_ID'].map(order)
features_df = features_df.sort_values('_order').drop(columns='_order')
features_df.to_csv(FINAL_CSV, index=False)

print(f'Final rows: {len(features_df):,} / {len(metadata):,}')
print(f'Missing rows: {len(missing_ids):,}')
print(f'Feature columns: {len(FEATURE_COLUMNS)}')
print(f'HNR missing: {features_df.hnr_mean_db.isna().sum():,}')
print(f'Inharmonicity missing: {features_df.inharmonicity_mean.isna().sum():,}')
print('Final CSV:', FINAL_CSV)
print('Configuration:', CONFIG_JSON)
print('Error log:', ERROR_CSV)

if missing_ids:
    print('Some tracks are not complete. Inspect the error log and rerun the extraction cell.')
else:
    print('All metadata tracks have one verified output row.')

display(features_df.head())
display(features_df[FEATURE_COLUMNS].describe().T)


## Output files

The notebook writes:

- `timbre_features/timbre_features_raw.csv` - final one-row-per-track feature table;
- `timbre_features/timbre_features_checkpoint.csv` - resumable extraction journal;
- `timbre_features/timbre_feature_errors.csv` - tracks that could not be processed;
- `timbre_features/timbre_extraction_config.json` - exact feature order and extraction parameters.

Use `timbre_features_raw.csv` as the source for `d_true`. During model training, fit the 35-feature standardizer on the training partition only and apply that saved transformation to validation and test tracks.
